In [ ]:
import os
import cv2
import glob
import random
import shutil
from multiprocessing import Pool, cpu_count
from tqdm import tqdm
from skimage.metrics import structural_similarity as ssim


In [ ]:
# Montar Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Ruta General del dataset
base_path = "/content/dataset_GAN"

# Folder divisivos
folders = ["train/live_action","train/cartoon","test/live_action","test/cartoon"]

# Crear carpetas
for folder in folders:
    os.makedirs(os.path.join(base_path, folder), exist_ok=True)

print("Estructura creada")

Estructura creada


In [ ]:
# Funcion para centrar la imágen antes de re-size
def center_crop(img):
    h, w, _ = img.shape
    min_dim = min(h, w)

    start_x = w // 2 - min_dim // 2
    start_y = h // 2 - min_dim // 2

    return img[start_y:start_y + min_dim, start_x:start_x + min_dim]

# Función para extraer las imágenes de los videos
def extract_frames(args):
    video_path, output_folder, frame_skip, resize = args

    cap = cv2.VideoCapture(video_path)

    # Variables contador
    saved = 0
    count = 0

    # Nombre del video sin extensiones
    video_name = os.path.splitext(os.path.basename(video_path))[0]

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # Solo guardar cada ciertos frames
        if count % frame_skip == 0:
            # Centrar y cortar
            frame = center_crop(frame)
            frame = cv2.resize(frame, resize, interpolation=cv2.INTER_AREA)

            # Guardar
            filename = os.path.join(output_folder, f"{video_name}_{saved:05d}.jpg")
            cv2.imwrite(filename, frame, [int(cv2.IMWRITE_JPEG_QUALITY), 90])

            saved += 1

        count += 1

    cap.release()
    print(f"{video_name} → {saved} frames")
    return saved

In [ ]:
# Funcion para procesar las imágenes en paralelo
def process_videos_parallel(input_folder, output_folder, frame_skip=20, resize=(256,256)):
    os.makedirs(output_folder, exist_ok=True)

    videos = glob.glob(input_folder + "/*.mp4")

    num_workers = min(4, cpu_count())
    print(f"Usando {num_workers} procesos")

    args_list = [(video, output_folder, frame_skip, resize) for video in videos]

    with Pool(num_workers) as p:
        # Usar vrios workers para más rápido
        results = list(tqdm(p.imap_unordered(extract_frames, args_list),total=len(args_list),desc="Procesando películas"))

    print(f"\nTotal imágenes: {sum(results)}")

#Live Action
process_videos_parallel("/content/drive/MyDrive/GAN/live_action","/content/dataset_GAN/train/live_action", frame_skip=15)

# Cartoon
process_videos_parallel("/content/drive/MyDrive/GAN/cartoon", "/content/dataset_GAN/train/cartoon", frame_skip=20)

Usando 2 procesos


Procesando películas:   0%|          | 0/5 [00:00<?, ?it/s]

Aladdin_LA → 238 frames


Procesando películas:  20%|██        | 1/5 [01:18<05:12, 78.23s/it]

BellaBestia2_LA → 568 frames


...

Aurora_bueno → 346 frames


Procesando películas:  80%|████████  | 4/5 [05:38<01:19, 79.52s/it]

Sapo_bueno → 296 frames


Procesando películas: 100%|██████████| 5/5 [05:42<00:00, 68.57s/it]


Total imágenes: 1829


In [ ]:
# Funcion para ver si los frames no son identicos
def remove_similar_images(folder, threshold=0.95):
    images = sorted(os.listdir(folder))

    prev_img = None
    removed = 0

    for img_name in images:
        img_path = os.path.join(folder, img_name)

        img = cv2.imread(img_path)
        if img is None:
            continue

        img = cv2.resize(img, (256, 256))
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        if prev_img is None:
            prev_img = gray
            continue

        score, _ = ssim(prev_img, gray, full=True)

        if score > threshold:
            os.remove(img_path)
            removed += 1
        else:
            prev_img = gray

    print(f"{folder} → eliminadas: {removed}")

In [ ]:
# Eliminar frames muy parecidos
remove_similar_images("/content/dataset_GAN/train/live_action", 0.95)
remove_similar_images("/content/dataset_GAN/train/cartoon", 0.95)

/content/dataset_GAN/train/live_action → eliminadas: 27
/content/dataset_GAN/train/cartoon → eliminadas: 114


In [ ]:
# Funcion para listar tamaño de una carpeta
def count_images(folder):
    return len(os.listdir(folder))

print("LIVE:", count_images("/content/dataset_GAN/train/live_action"))
print("CARTOON:", count_images("/content/dataset_GAN/train/cartoon"))

LIVE: 1663
CARTOON: 1715


In [ ]:
# Hacemos train/test split
def split_train_test(source_folder, test_folder, split=0.1, seed=42):
    os.makedirs(test_folder, exist_ok=True)

    images = os.listdir(source_folder)
    test_size = int(len(images) * split)

    rng = random.Random(seed)
    test_images = rng.sample(images, test_size)

    for img in test_images:
        shutil.move(os.path.join(source_folder, img),os.path.join(test_folder, img))

    print(f"{test_size} imágenes movidas a {test_folder}")

# LA
split_train_test("/content/dataset_GAN/train/live_action","/content/dataset_GAN/test/live_action",0.1)

# Cartoon
split_train_test("/content/dataset_GAN/train/cartoon","/content/dataset_GAN/test/cartoon",0.1)

149 imágenes movidas a /content/dataset_GAN/test/live_action
154 imágenes movidas a /content/dataset_GAN/test/cartoon


In [ ]:
print("TRAIN LIVE:", count_images("/content/dataset_GAN/train/live_action"))
print("TRAIN CARTOON:", count_images("/content/dataset_GAN/train/cartoon"))
print("TEST LIVE:", count_images("/content/dataset_GAN/test/live_action"))
print("TEST CARTOON:", count_images("/content/dataset_GAN/test/cartoon"))

TRAIN LIVE: 1497
TRAIN CARTOON: 1544
TEST LIVE: 166
TEST CARTOON: 171


In [ ]:
# Ver tamaño del archivo
!du -sh /content/dataset_GAN

55M	/content/dataset_GAN


In [ ]:
# Comprimir en un archivo zip para más facil de manejar
shutil.make_archive("/content/drive/MyDrive/GAN/dataset_GAN",'zip',"/content/dataset_GAN")
print("Zip guardado en Drive")

Zip guardado en Drive
